# 1. Inspect dataset files

Verify that the downloaded validation images and labels can be read correctly.

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
cwd = Path.cwd().resolve()
project_root = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / "data" / "val.X").exists() and (candidate / "data" / "Labels.json").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("Could not find project root containing data/val.X and data/Labels.json")

data_dir = project_root / "data"
val_path = data_dir / "val.X"
labels_path = data_dir / "Labels.json"

print(f"current working directory: {cwd}")
print(f"project_root: {project_root}")
print(f"data_dir: {data_dir}")
print("Files/folders inside data/:")
for path in sorted(data_dir.iterdir()):
    print(path.name)

In [ ]:
print(f"val.X exists: {val_path.exists()}")
print(f"val.X is folder: {val_path.is_dir()}")
print(f"val.X is file: {val_path.is_file()}")

image_extensions = {".jpg", ".jpeg", ".png"}

if val_path.is_dir():
    image_paths = sorted(
        path for path in val_path.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
elif val_path.is_file() and val_path.suffix.lower() in image_extensions:
    image_paths = [val_path]
else:
    image_paths = []

print(f"Number of images found: {len(image_paths)}")
print("First 5 image paths:")
for path in image_paths[:5]:
    print(path)

In [ ]:
with labels_path.open("r") as f:
    labels = json.load(f)

print(f"Labels type: {type(labels)}")

if isinstance(labels, dict):
    print("First few keys:")
    for key in list(labels.keys())[:5]:
        print(key)
elif isinstance(labels, list):
    print("First few items:")
    for item in labels[:5]:
        print(item)
else:
    print("Label preview:")
    print(labels)

In [ ]:
first_image_path = image_paths[0]

image = Image.open(first_image_path).convert("RGB")
print(f"First image path: {first_image_path}")
print(f"Image size: {image.size}")

plt.imshow(image)
plt.axis("off")
plt.show()

# 2. Apply ViT-B/16 preprocessing

We use `weights.transforms()` because the preprocessing must match the pretrained ViT-B/16 weights.

In [ ]:
import torch
from torchvision.models import vit_b_16, ViT_B_16_Weights

weights = ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

preprocessed_img = preprocess(image)
x = preprocessed_img.unsqueeze(0)

print(f"Original PIL image size: {image.size}")
print(f"Preprocessed tensor shape: {preprocessed_img.shape}")
print(f"Batched tensor shape: {x.shape}")
print(f"Tensor dtype: {x.dtype}")
print(f"Tensor min value: {x.min().item()}")
print(f"Tensor max value: {x.max().item()}")

assert list(x.shape) == [1, 3, 224, 224]

# 3. Run ViT-B/16 forward pass

The model output is a 1000-dimensional ImageNet logits vector. We check top-5 predictions only as a sanity check before extracting activations.

In [ ]:
import torch

model = vit_b_16(weights=weights)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
x = x.to(device)

with torch.no_grad():
    logits = model(x)

print(f"Device: {device}")
print(f"Logits shape: {logits.shape}")
print(f"Logits dtype: {logits.dtype}")

assert list(logits.shape) == [1, 1000]

probs = torch.softmax(logits, dim=1)
top5_prob, top5_idx = probs.topk(5, dim=1)
categories = weights.meta["categories"]

print("Top-5 predictions:")
for rank, (class_idx, prob) in enumerate(zip(top5_idx[0], top5_prob[0]), start=1):
    class_idx = class_idx.item()
    prob = prob.item()
    class_name = categories[class_idx]
    print(f"{rank}. index={class_idx}, class={class_name}, probability={prob:.4f}")

# 4. Extract layer 6 activations with a forward hook

The hooked activation has shape `[B, 197, 768]`, where 197 = 1 CLS token + 196 patch tokens.

In [ ]:
print(model.encoder)

print(f"type(model.encoder.layers): {type(model.encoder.layers)}")
print(f"Number of transformer layers: {len(model.encoder.layers)}")

print("First few layer names:")
for idx, name in enumerate(model.encoder.layers._modules.keys()):
    if idx >= 5:
        break
    print(f"{idx}: {name}")

In [ ]:
activations = {}
target_layer = model.encoder.layers[5]

def save_layer_6_activation(module, inputs, output):
    activations["layer_6"] = output.detach().cpu()

hook_handle = target_layer.register_forward_hook(save_layer_6_activation)

with torch.no_grad():
    _ = model(x)

hook_handle.remove()

activation = activations["layer_6"]

print(f"Activation shape: {activation.shape}")
print(f"Activation dtype: {activation.dtype}")
print("Expected shape: [B, 197, 768], where B=1, 197 = 1 CLS token + 196 patch tokens, and 768 is ViT-B/16 hidden size.")

assert activation.ndim == 3
assert activation.shape[0] == 1
assert activation.shape[1] == 197
assert activation.shape[2] == 768

# 5. Separate CLS token and patch tokens

We focus on patch tokens because each patch token corresponds to a spatial region of the image. This allows us to later map SAE feature activations back to image patches.

In [ ]:
activation = activations["layer_6"]

cls_token = activation[:, 0, :]
patch_tokens = activation[:, 1:, :]

print(f"CLS token shape: {cls_token.shape}")
print(f"Patch tokens shape: {patch_tokens.shape}")

assert list(cls_token.shape) == [1, 768]
assert list(patch_tokens.shape) == [1, 196, 768]

patch_vectors = patch_tokens.reshape(-1, patch_tokens.shape[-1])

print(f"Flattened patch vectors shape: {patch_vectors.shape}")

assert list(patch_vectors.shape) == [196, 768]

# 6. Save layer 6 patch-token activation sample and metadata

The metadata is necessary because later, when a SAE feature strongly activates on a vector, we need to map that vector back to the original image and patch position.

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = project_root / "outputs" / "activations"
output_dir.mkdir(parents=True, exist_ok=True)

activation_path = output_dir / "layer_06_patch_tokens_sample.pt"
metadata_path = output_dir / "layer_06_metadata_sample.csv"

torch.save(patch_tokens, activation_path)

class_id = first_image_path.parent.name if first_image_path.parent.name in labels else None
class_name = labels.get(class_id) if class_id is not None else None

metadata_rows = []
for patch_id in range(patch_tokens.shape[1]):
    metadata_rows.append({
        "image_index": 0,
        "image_path": str(first_image_path),
        "patch_id": patch_id,
        "patch_row": patch_id // 14,
        "patch_col": patch_id % 14,
        "class_id": class_id,
        "class_name": class_name,
    })

metadata = pd.DataFrame(metadata_rows)
metadata.to_csv(metadata_path, index=False)

loaded_patch_tokens = torch.load(activation_path)

print(f"Saved activation path: {activation_path}")
print(f"Saved metadata path: {metadata_path}")
print("Metadata head:")
display(metadata.head())
print(f"Loaded activation shape: {loaded_patch_tokens.shape}")

assert list(loaded_patch_tokens.shape) == [1, 196, 768]
assert len(metadata) == 196

# 7. Extract layer 6 patch-token activations for a small image batch

In [ ]:
max_images = 32
selected_image_paths = image_paths[:max_images]

batch_images = []
for path in selected_image_paths:
    img = Image.open(path).convert("RGB")
    batch_images.append(preprocess(img))

batch_x = torch.stack(batch_images, dim=0)

activations = {}
target_layer = model.encoder.layers[5]

def save_layer_6_batch_activation(module, inputs, output):
    activations["layer_6_batch"] = output.detach().cpu()

hook_handle = target_layer.register_forward_hook(save_layer_6_batch_activation)

batch_x = batch_x.to(device)
with torch.no_grad():
    _ = model(batch_x)

hook_handle.remove()

activation = activations["layer_6_batch"]
patch_tokens = activation[:, 1:, :]

print(f"batch_x shape: {batch_x.shape}")
print(f"activation shape: {activation.shape}")
print(f"patch_tokens shape: {patch_tokens.shape}")

assert list(batch_x.shape) == [max_images, 3, 224, 224]
assert list(activation.shape) == [max_images, 197, 768]
assert list(patch_tokens.shape) == [max_images, 196, 768]

metadata_rows = []
for image_index, image_path in enumerate(selected_image_paths):
    class_id = image_path.parent.name if image_path.parent.name in labels else None
    class_name = labels.get(class_id) if class_id is not None else None

    for patch_id in range(196):
        metadata_rows.append({
            "image_index": image_index,
            "image_path": str(image_path),
            "patch_id": patch_id,
            "patch_row": patch_id // 14,
            "patch_col": patch_id % 14,
            "class_id": class_id,
            "class_name": class_name,
        })

metadata = pd.DataFrame(metadata_rows)

activation_path = output_dir / "layer_06_patch_tokens_32.pt"
metadata_path = output_dir / "layer_06_metadata_32.csv"

torch.save(patch_tokens, activation_path)
metadata.to_csv(metadata_path, index=False)

loaded_patch_tokens = torch.load(activation_path)

print(f"Saved activation path: {activation_path}")
print(f"Saved metadata path: {metadata_path}")
print(f"Loaded saved tensor shape: {loaded_patch_tokens.shape}")

assert list(loaded_patch_tokens.shape) == [32, 196, 768]
assert len(metadata) == 32 * 196

# 8. Train a toy SAE on layer 6 patch-token activations

This toy SAE is only for testing the training pipeline. Since it uses only 32 images, the learned features are not expected to be meaningful yet.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

activation_path = project_root / "outputs" / "activations" / "layer_06_patch_tokens_32.pt"
patch_tokens = torch.load(activation_path)
acts = patch_tokens.reshape(-1, 768)

print(f"Original activation shape: {patch_tokens.shape}")
print(f"Flattened activation shape: {acts.shape}")

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=2048):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

d_in = 768
d_sae = 2048
l1_coeff = 1e-4
batch_size = 512
num_epochs = 10

sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)
optimizer = torch.optim.Adam(sae.parameters(), lr=1e-3)

dataset = TensorDataset(acts.float())
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

sae.train()
for epoch in range(num_epochs):
    total_loss_sum = 0.0
    reconstruction_loss_sum = 0.0
    sparsity_loss_sum = 0.0
    num_examples = 0

    for (batch,) in dataloader:
        batch = batch.to(device)

        x_hat, z = sae(batch)
        reconstruction_loss = F.mse_loss(x_hat, batch)
        sparsity_loss = z.abs().mean()
        total_loss = reconstruction_loss + l1_coeff * sparsity_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        batch_size_actual = batch.shape[0]
        total_loss_sum += total_loss.item() * batch_size_actual
        reconstruction_loss_sum += reconstruction_loss.item() * batch_size_actual
        sparsity_loss_sum += sparsity_loss.item() * batch_size_actual
        num_examples += batch_size_actual

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"total_loss={total_loss_sum / num_examples:.6f} | "
        f"reconstruction_loss={reconstruction_loss_sum / num_examples:.6f} | "
        f"sparsity_loss={sparsity_loss_sum / num_examples:.6f}"
    )

checkpoint_dir = project_root / "outputs" / "sae_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "toy_sae_layer_06.pt"

torch.save({
    "model_state_dict": sae.state_dict(),
    "d_in": d_in,
    "d_sae": d_sae,
    "l1_coeff": l1_coeff,
    "num_epochs": num_epochs,
    "activation_path": str(activation_path),
}, checkpoint_path)

print(f"Saved SAE checkpoint: {checkpoint_path}")

# 9. Rank top activating patches for SAE features

For each SAE feature, we rank all patch vectors by the feature activation coefficient. The highest-ranking patches are the visual evidence used to interpret what that SAE feature represents.

In [ ]:
import pandas as pd

activation_path = project_root / "outputs" / "activations" / "layer_06_patch_tokens_32.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_06_metadata_32.csv"
checkpoint_path = project_root / "outputs" / "sae_checkpoints" / "toy_sae_layer_06.pt"

patch_tokens = torch.load(activation_path)
acts = patch_tokens.reshape(-1, 768)
metadata = pd.read_csv(metadata_path)

assert acts.shape[0] == len(metadata)
assert acts.shape[1] == 768

d_in = 768
d_sae = 2048

sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

with torch.no_grad():
    z = F.relu(sae.encoder(acts.float().to(device))).cpu()

print(f"acts shape: {acts.shape}")
print(f"z shape: {z.shape}")
print("Expected z shape: [6272, 2048]")

feature_ids = [0, 1, 2, 10, 100]
top_k = 10
ranking_rows = []

for feature_id in feature_ids:
    feature_activations = z[:, feature_id]
    top_values, top_indices = torch.topk(feature_activations, k=top_k)

    feature_rows = metadata.iloc[top_indices.tolist()].copy()
    feature_rows.insert(0, "feature_id", feature_id)
    feature_rows.insert(1, "rank", range(1, top_k + 1))
    feature_rows["feature_activation"] = top_values.tolist()

    ranking_rows.append(feature_rows)

    print(f"Top {top_k} patches for feature {feature_id}:")
    display(feature_rows[[
        "rank",
        "image_index",
        "image_path",
        "patch_id",
        "patch_row",
        "patch_col",
        "class_id",
        "class_name",
        "feature_activation",
    ]])

top_patches = pd.concat(ranking_rows, ignore_index=True)

ranking_dir = project_root / "outputs" / "feature_rankings"
ranking_dir.mkdir(parents=True, exist_ok=True)
ranking_path = ranking_dir / "toy_layer06_top_patches.csv"
top_patches.to_csv(ranking_path, index=False)

print(f"Saved top-k feature rankings: {ranking_path}")

# 10. Visualize top activating image patches

Since ViT-B/16 uses 16x16 patches on 224x224 images, each patch token corresponds to one 16x16 image region. Visualizing the top activating patches helps us interpret what a SAE feature represents.

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

ranking_path = project_root / "outputs" / "feature_rankings" / "toy_layer06_top_patches.csv"
rankings = pd.read_csv(ranking_path)

feature_id = 0
feature_rankings = rankings[rankings["feature_id"] == feature_id]
top_patches = feature_rankings.sort_values("feature_activation", ascending=False).head(10)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for ax, (_, row) in zip(axes, top_patches.iterrows()):
    image = Image.open(row["image_path"]).convert("RGB")
    image = image.resize((224, 224))

    patch_row = int(row["patch_row"])
    patch_col = int(row["patch_col"])

    left = patch_col * 16
    upper = patch_row * 16
    right = left + 16
    lower = upper + 16

    patch = image.crop((left, upper, right, lower))

    ax.imshow(patch)
    ax.set_title(f"act={row['feature_activation']:.3f}\nr={patch_row}, c={patch_col}")
    ax.axis("off")

for ax in axes[len(top_patches):]:
    ax.axis("off")

plt.suptitle(f"Top activating patches for SAE feature {feature_id}")
plt.tight_layout()
plt.show()

# 11. Find the most active SAE features

Instead of inspecting arbitrary feature ids, we first identify features that are actually active on the dataset. Highly active features are better candidates for visualization and interpretation.

In [ ]:
mean_activation = z.mean(dim=0)
max_activation = z.max(dim=0).values
activation_frequency = (z > 0).float().mean(dim=0)

feature_stats = pd.DataFrame({
    "feature_id": range(z.shape[1]),
    "mean_activation": mean_activation.numpy(),
    "max_activation": max_activation.numpy(),
    "activation_frequency": activation_frequency.numpy(),
})

feature_stats = feature_stats.sort_values("max_activation", ascending=False).reset_index(drop=True)

print("Top 20 most active SAE features by max activation:")
display(feature_stats.head(20))

feature_stats_path = project_root / "outputs" / "feature_rankings" / "toy_layer06_feature_stats.csv"
feature_stats.to_csv(feature_stats_path, index=False)

top_5_feature_ids = feature_stats.head(5)["feature_id"].tolist()
print(f"Top 5 feature ids by max activation: {top_5_feature_ids}")
print(f"Saved feature stats: {feature_stats_path}")

# 12. Visualize the top 5 most active SAE features

We visualize the most active SAE features because arbitrary features may be inactive or uninterpretable. The top patches help us inspect whether a feature corresponds to a coherent visual pattern.

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

ranking_path = project_root / "outputs" / "feature_rankings" / "toy_layer06_top_patches.csv"
rankings = pd.read_csv(ranking_path)

top_feature_ids = [566, 643, 54, 1128, 1546]
top_k = 10

missing_feature_ids = [
    feature_id for feature_id in top_feature_ids
    if feature_id not in set(rankings["feature_id"].unique())
]

if missing_feature_ids:
    metadata_path = project_root / "outputs" / "activations" / "layer_06_metadata_32.csv"
    metadata = pd.read_csv(metadata_path)

    new_ranking_rows = []
    for feature_id in missing_feature_ids:
        feature_activations = z[:, feature_id]
        top_values, top_indices = torch.topk(feature_activations, k=top_k)

        feature_rows = metadata.iloc[top_indices.tolist()].copy()
        feature_rows.insert(0, "feature_id", feature_id)
        feature_rows.insert(1, "rank", range(1, top_k + 1))
        feature_rows["feature_activation"] = top_values.cpu().tolist()
        new_ranking_rows.append(feature_rows)

    rankings = pd.concat([rankings, *new_ranking_rows], ignore_index=True)
    rankings.to_csv(ranking_path, index=False)

visualization_dir = project_root / "outputs" / "feature_visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)

for feature_id in top_feature_ids:
    feature_rankings = rankings[rankings["feature_id"] == feature_id]
    top_patches = feature_rankings.sort_values("feature_activation", ascending=False).head(top_k)

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, top_patches.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")
        image = image.resize((224, 224))

        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])

        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16

        patch = image.crop((left, upper, right, lower))

        ax.imshow(patch)
        ax.set_title(f"act={row['feature_activation']:.3f}\nr={patch_row}, c={patch_col}")
        ax.axis("off")

    for ax in axes[len(top_patches):]:
        ax.axis("off")

    fig.suptitle(f"Top activating patches for SAE feature {feature_id}")
    fig.tight_layout()

    save_path = visualization_dir / f"toy_layer06_feature_{feature_id}_top_patches.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Saved visualization: {save_path}")

    plt.show()

# 13. Sample a more diverse image subset

A diverse subset is important because SAE features should capture general visual concepts rather than patterns from a single ImageNet class.

In [ ]:
from collections import defaultdict

num_classes = 5
images_per_class = 10

print("Example image paths:")
for path in image_paths[:5]:
    print(path)

# Images are stored under data/val.X/<class_id>/<image_file>, so the parent folder is the class id.
images_by_class = defaultdict(list)
for path in image_paths:
    class_id = path.parent.name
    if class_id in labels:
        images_by_class[class_id].append(path)

eligible_class_ids = [
    class_id for class_id, paths in images_by_class.items()
    if len(paths) >= images_per_class
]

if len(eligible_class_ids) >= num_classes:
    selected_class_ids = eligible_class_ids[:num_classes]
    selected_rows = []

    for class_id in selected_class_ids:
        selected_paths = images_by_class[class_id][:images_per_class]
        for path in selected_paths:
            selected_rows.append({
                "image_path": str(path),
                "class_id": class_id,
                "class_name": labels.get(class_id),
            })
else:
    print("Could not infer enough classes from path structure; falling back to evenly spaced image paths.")
    step = max(1, len(image_paths) // (num_classes * images_per_class))
    fallback_paths = image_paths[::step][:num_classes * images_per_class]
    selected_rows = []

    for path in fallback_paths:
        class_id = path.parent.name if path.parent.name in labels else None
        selected_rows.append({
            "image_path": str(path),
            "class_id": class_id,
            "class_name": labels.get(class_id) if class_id is not None else None,
        })

diverse_subset = pd.DataFrame(selected_rows)
diverse_subset.insert(0, "image_index", range(len(diverse_subset)))

selected_class_ids = diverse_subset["class_id"].dropna().unique().tolist()
class_counts = diverse_subset["class_id"].value_counts(dropna=False)

subset_path = project_root / "outputs" / "activations" / "diverse_subset_50.csv"
diverse_subset.to_csv(subset_path, index=False)

print(f"Number of selected images: {len(diverse_subset)}")
print(f"Selected class ids: {selected_class_ids}")
print("Number of images per selected class:")
print(class_counts)
print("First few selected image paths:")
for path in diverse_subset["image_path"].head().tolist():
    print(path)
print(f"Saved diverse subset: {subset_path}")

assert len(diverse_subset) == num_classes * images_per_class

# 14. Extract layer 6 activations for the diverse subset

In [ ]:
subset_path = project_root / "outputs" / "activations" / "diverse_subset_50.csv"
diverse_subset = pd.read_csv(subset_path)

selected_image_paths = diverse_subset["image_path"].tolist()

batch_images = []
for path in selected_image_paths:
    img = Image.open(path).convert("RGB")
    batch_images.append(preprocess(img))

batch_x = torch.stack(batch_images, dim=0)

activations = {}
target_layer = model.encoder.layers[5]

def save_layer_6_diverse_activation(module, inputs, output):
    activations["layer_6_diverse"] = output.detach().cpu()

hook_handle = target_layer.register_forward_hook(save_layer_6_diverse_activation)

batch_x = batch_x.to(device)
with torch.no_grad():
    _ = model(batch_x)

hook_handle.remove()

activation = activations["layer_6_diverse"]
patch_tokens = activation[:, 1:, :]

print(f"batch_x shape: {batch_x.shape}")
print(f"activation shape: {activation.shape}")
print(f"patch_tokens shape: {patch_tokens.shape}")

assert list(batch_x.shape) == [50, 3, 224, 224]
assert list(activation.shape) == [50, 197, 768]
assert list(patch_tokens.shape) == [50, 196, 768]

activation_path = project_root / "outputs" / "activations" / "layer_06_patch_tokens_diverse_50.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_06_metadata_diverse_50.csv"

torch.save(patch_tokens, activation_path)

metadata_rows = []
for _, image_row in diverse_subset.iterrows():
    for patch_id in range(196):
        metadata_rows.append({
            "image_index": int(image_row["image_index"]),
            "image_path": image_row["image_path"],
            "class_id": image_row["class_id"],
            "class_name": image_row["class_name"],
            "patch_id": patch_id,
            "patch_row": patch_id // 14,
            "patch_col": patch_id % 14,
        })

metadata = pd.DataFrame(metadata_rows)
metadata.to_csv(metadata_path, index=False)

assert len(metadata) == 50 * 196

loaded_patch_tokens = torch.load(activation_path)

print(f"Saved activation path: {activation_path}")
print(f"Saved metadata path: {metadata_path}")
print(f"Loaded saved tensor shape: {loaded_patch_tokens.shape}")

assert list(loaded_patch_tokens.shape) == [50, 196, 768]

# 15. Train a SAE on the diverse 50-image subset

This SAE is trained on a slightly more diverse 50-image subset. It is still a small-scale experiment, but it should be more informative than the previous 32-image single-class toy run.

In [ ]:
activation_path = project_root / "outputs" / "activations" / "layer_06_patch_tokens_diverse_50.pt"
patch_tokens = torch.load(activation_path)
acts = patch_tokens.reshape(-1, 768)

print(f"Original activation shape: {patch_tokens.shape}")
print(f"Flattened activation shape: {acts.shape}")

d_in = 768
d_sae = 2048
l1_coeff = 1e-3
batch_size = 512
num_epochs = 20
learning_rate = 1e-3

sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)
optimizer = torch.optim.Adam(sae.parameters(), lr=learning_rate)

dataset = TensorDataset(acts.float())
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

sae.train()
for epoch in range(num_epochs):
    total_loss_sum = 0.0
    reconstruction_loss_sum = 0.0
    sparsity_loss_sum = 0.0
    num_examples = 0

    for (batch,) in dataloader:
        batch = batch.to(device)

        x_hat, z = sae(batch)
        reconstruction_loss = F.mse_loss(x_hat, batch)
        sparsity_loss = z.abs().mean()
        total_loss = reconstruction_loss + l1_coeff * sparsity_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        batch_size_actual = batch.shape[0]
        total_loss_sum += total_loss.item() * batch_size_actual
        reconstruction_loss_sum += reconstruction_loss.item() * batch_size_actual
        sparsity_loss_sum += sparsity_loss.item() * batch_size_actual
        num_examples += batch_size_actual

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"total_loss={total_loss_sum / num_examples:.6f} | "
        f"reconstruction_loss={reconstruction_loss_sum / num_examples:.6f} | "
        f"sparsity_loss={sparsity_loss_sum / num_examples:.6f}"
    )

checkpoint_dir = project_root / "outputs" / "sae_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "sae_layer06_diverse50.pt"

torch.save({
    "model_state_dict": sae.state_dict(),
    "d_in": d_in,
    "d_sae": d_sae,
    "l1_coeff": l1_coeff,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "learning_rate": learning_rate,
    "activation_path": str(activation_path),
}, checkpoint_path)

print(f"Saved SAE checkpoint: {checkpoint_path}")

# 16. Rank and visualize top features from the diverse-50 SAE

We now inspect the most active features from the SAE trained on the more diverse 50-image subset. If top patches share visual patterns, this suggests that the SAE feature may correspond to an interpretable visual concept.

In [ ]:
activation_path = project_root / "outputs" / "activations" / "layer_06_patch_tokens_diverse_50.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_06_metadata_diverse_50.csv"
checkpoint_path = project_root / "outputs" / "sae_checkpoints" / "sae_layer06_diverse50.pt"

patch_tokens = torch.load(activation_path)
acts = patch_tokens.reshape(-1, 768)
metadata = pd.read_csv(metadata_path)

assert len(metadata) == acts.shape[0]

d_in = 768
d_sae = 2048

sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

with torch.no_grad():
    z = F.relu(sae.encoder(acts.float().to(device))).cpu()

print(f"acts shape: {acts.shape}")
print(f"z shape: {z.shape}")

mean_activation = z.mean(dim=0)
max_activation = z.max(dim=0).values
activation_frequency = (z > 0).float().mean(dim=0)

feature_stats = pd.DataFrame({
    "feature_id": range(z.shape[1]),
    "mean_activation": mean_activation.numpy(),
    "max_activation": max_activation.numpy(),
    "activation_frequency": activation_frequency.numpy(),
})
feature_stats = feature_stats.sort_values("max_activation", ascending=False).reset_index(drop=True)

ranking_dir = project_root / "outputs" / "feature_rankings"
ranking_dir.mkdir(parents=True, exist_ok=True)
feature_stats_path = ranking_dir / "layer06_diverse50_feature_stats.csv"
feature_stats.to_csv(feature_stats_path, index=False)

top_feature_ids = feature_stats.head(5)["feature_id"].tolist()
print(f"Top 5 feature ids by max activation: {top_feature_ids}")
print(f"Saved feature stats: {feature_stats_path}")

top_k = 10
top_patch_rows = []

for feature_id in top_feature_ids:
    feature_activations = z[:, feature_id]
    top_values, top_indices = torch.topk(feature_activations, k=top_k)

    feature_rows = metadata.iloc[top_indices.tolist()].copy()
    feature_rows.insert(0, "feature_id", feature_id)
    feature_rows.insert(1, "rank", range(1, top_k + 1))
    feature_rows["feature_activation"] = top_values.tolist()
    top_patch_rows.append(feature_rows)

top_patches = pd.concat(top_patch_rows, ignore_index=True)
top_patches_path = ranking_dir / "layer06_diverse50_top_patches.csv"
top_patches.to_csv(top_patches_path, index=False)
print(f"Saved top patch rankings: {top_patches_path}")

visualization_dir = project_root / "outputs" / "feature_visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)

for feature_id in top_feature_ids:
    feature_patches = top_patches[top_patches["feature_id"] == feature_id]

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, feature_patches.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")
        image = image.resize((224, 224))

        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])

        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16

        patch = image.crop((left, upper, right, lower))

        ax.imshow(patch)
        ax.set_title(f"act={row['feature_activation']:.3f}\nr={patch_row}, c={patch_col}")
        ax.axis("off")

    for ax in axes[len(feature_patches):]:
        ax.axis("off")

    fig.suptitle(f"Top activating patches for diverse-50 SAE feature {feature_id}")
    fig.tight_layout()

    visualization_path = visualization_dir / f"layer06_diverse50_feature_{feature_id}_top_patches.png"
    fig.savefig(visualization_path, dpi=150, bbox_inches="tight")
    print(f"Saved visualization: {visualization_path}")

    plt.show()

# Visualize top patches in full image context

In [ ]:
from PIL import Image, ImageDraw

ranking_path = project_root / "outputs" / "feature_rankings" / "layer06_diverse50_top_patches.csv"
rankings = pd.read_csv(ranking_path)

feature_id = 1969
feature_rankings = rankings[rankings["feature_id"] == feature_id]
top_patches = feature_rankings.sort_values("feature_activation", ascending=False).head(5)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, (_, row) in zip(axes, top_patches.iterrows()):
    image = Image.open(row["image_path"]).convert("RGB")
    image = image.resize((224, 224))

    patch_row = int(row["patch_row"])
    patch_col = int(row["patch_col"])

    left = patch_col * 16
    upper = patch_row * 16
    right = left + 16
    lower = upper + 16

    draw = ImageDraw.Draw(image)
    draw.rectangle((left, upper, right, lower), outline="red", width=3)

    ax.imshow(image)
    ax.set_title(
        f"feature={feature_id}\n"
        f"act={row['feature_activation']:.3f}\n"
        f"r={patch_row}, c={patch_col}\n"
        f"{row['class_name']}"
    )
    ax.axis("off")

for ax in axes[len(top_patches):]:
    ax.axis("off")

fig.suptitle(f"Feature {feature_id}: top activating patches in full image context")
fig.tight_layout()

visualization_path = project_root / "outputs" / "feature_visualizations" / "layer06_diverse50_feature_1969_full_image_context.png"
fig.savefig(visualization_path, dpi=150, bbox_inches="tight")
print(f"Saved full image context visualization: {visualization_path}")

plt.show()

# Sample a truly diverse ImageNet subset

We select classes evenly across the ImageNet class list to avoid choosing visually similar neighboring classes. This should produce a more diverse subset for SAE training.

In [ ]:
import numpy as np

num_classes = 20
images_per_class = 10
random_state = 42

image_rows = []
for path in image_paths:
    class_id = path.parent.name
    image_rows.append({
        "image_path": str(path),
        "class_id": class_id,
        "class_name": labels.get(class_id),
    })

image_df = pd.DataFrame(image_rows)
images_by_class = image_df.groupby("class_id")

class_ids = sorted(image_df["class_id"].unique().tolist())
eligible_class_ids = [
    class_id for class_id in class_ids
    if len(images_by_class.get_group(class_id)) >= images_per_class
]

selected_class_indices = np.linspace(0, len(eligible_class_ids) - 1, num_classes, dtype=int)
selected_class_ids = [eligible_class_ids[idx] for idx in selected_class_indices]

selected_dfs = []
for class_id in selected_class_ids:
    class_df = images_by_class.get_group(class_id)
    selected_dfs.append(class_df.sample(n=images_per_class, random_state=random_state))

diverse_subset_200 = pd.concat(selected_dfs, ignore_index=True)
diverse_subset_200.insert(0, "image_index", range(len(diverse_subset_200)))
diverse_subset_200 = diverse_subset_200[["image_index", "image_path", "class_id", "class_name"]]

class_counts = diverse_subset_200["class_id"].value_counts().loc[selected_class_ids]
selected_class_names = [labels.get(class_id) for class_id in selected_class_ids]

subset_path = project_root / "outputs" / "activations" / "diverse_subset_200.csv"
diverse_subset_200.to_csv(subset_path, index=False)

print(f"Total number of classes found: {len(class_ids)}")
print(f"Selected class ids: {selected_class_ids}")
print(f"Selected class names: {selected_class_names}")
print("Number of images per selected class:")
print(class_counts)
print(f"Total selected images: {len(diverse_subset_200)}")
print("First 10 selected image paths:")
for path in diverse_subset_200["image_path"].head(10).tolist():
    print(path)
print(f"Saved diverse subset: {subset_path}")

assert len(selected_class_ids) == num_classes
assert len(diverse_subset_200) == num_classes * images_per_class

# Extract layer 6 activations for diverse subset 200

We process images in mini-batches to avoid memory issues. The saved patch-token activations will be used to train a stronger SAE.

In [ ]:
subset_path = project_root / "outputs" / "activations" / "diverse_subset_200.csv"
diverse_subset_200 = pd.read_csv(subset_path)

image_paths_200 = diverse_subset_200["image_path"].tolist()
batch_size = 32

target_layer = model.encoder.layers[5]
batch_activations = {}

def save_layer_6_batch_activation(module, inputs, output):
    batch_activations["layer_6"] = output.detach().cpu()

hook_handle = target_layer.register_forward_hook(save_layer_6_batch_activation)

patch_token_batches = []

for start_idx in range(0, len(image_paths_200), batch_size):
    end_idx = min(start_idx + batch_size, len(image_paths_200))
    batch_paths = image_paths_200[start_idx:end_idx]

    batch_images = []
    for path in batch_paths:
        img = Image.open(path).convert("RGB")
        batch_images.append(preprocess(img))

    batch_x = torch.stack(batch_images, dim=0).to(device)

    with torch.no_grad():
        _ = model(batch_x)

    activation = batch_activations["layer_6"]
    patch_tokens_batch = activation[:, 1:, :].cpu()
    patch_token_batches.append(patch_tokens_batch)

    print(f"Processed images {start_idx} to {end_idx - 1}: patch_tokens shape {patch_tokens_batch.shape}")

hook_handle.remove()

patch_tokens = torch.cat(patch_token_batches, dim=0)

metadata_rows = []
for _, image_row in diverse_subset_200.iterrows():
    for patch_id in range(196):
        metadata_rows.append({
            "image_index": int(image_row["image_index"]),
            "image_path": image_row["image_path"],
            "class_id": image_row["class_id"],
            "class_name": image_row["class_name"],
            "patch_id": patch_id,
            "patch_row": patch_id // 14,
            "patch_col": patch_id % 14,
        })

metadata = pd.DataFrame(metadata_rows)

activation_path = project_root / "outputs" / "activations" / "layer_06_patch_tokens_diverse_200.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_06_metadata_diverse_200.csv"

torch.save(patch_tokens, activation_path)
metadata.to_csv(metadata_path, index=False)

print(f"Number of images processed: {len(image_paths_200)}")
print(f"Final patch_tokens shape: {patch_tokens.shape}")
print(f"Metadata rows: {len(metadata)}")
print(f"Saved activation path: {activation_path}")
print(f"Saved metadata path: {metadata_path}")

assert list(patch_tokens.shape) == [200, 196, 768]
assert len(metadata) == 200 * 196

# Train SAE on diverse subset 200

This SAE is trained on a more diverse 200-image subset, producing 39,200 patch-token activation vectors. It is still smaller than the final experiment, but more reliable than the previous 32-image and 50-image toy runs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

activation_path = project_root / "outputs" / "activations" / "layer_06_patch_tokens_diverse_200.pt"
patch_tokens = torch.load(activation_path, map_location="cpu")
acts = patch_tokens.reshape(-1, 768)

print(f"Original activation shape: {patch_tokens.shape}")
print(f"Flattened activation shape: {acts.shape}")

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=4096):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

d_in = 768
d_sae = 4096
l1_coeff = 3e-3
batch_size = 1024
num_epochs = 30
learning_rate = 1e-3

sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)
dataset = TensorDataset(acts.float())
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
optimizer = torch.optim.Adam(sae.parameters(), lr=learning_rate)

metrics = []

for epoch in range(num_epochs):
    sae.train()
    total_loss_sum = 0.0
    reconstruction_loss_sum = 0.0
    sparsity_loss_sum = 0.0
    num_examples = 0

    for (batch_acts,) in dataloader:
        batch_acts = batch_acts.to(device)

        optimizer.zero_grad()
        x_hat, z = sae(batch_acts)
        reconstruction_loss = F.mse_loss(x_hat, batch_acts)
        sparsity_loss = z.abs().mean()
        total_loss = reconstruction_loss + l1_coeff * sparsity_loss
        total_loss.backward()
        optimizer.step()

        current_batch_size = batch_acts.shape[0]
        total_loss_sum += total_loss.item() * current_batch_size
        reconstruction_loss_sum += reconstruction_loss.item() * current_batch_size
        sparsity_loss_sum += sparsity_loss.item() * current_batch_size
        num_examples += current_batch_size

    epoch_total_loss = total_loss_sum / num_examples
    epoch_reconstruction_loss = reconstruction_loss_sum / num_examples
    epoch_sparsity_loss = sparsity_loss_sum / num_examples

    metrics.append({
        "epoch": epoch + 1,
        "total_loss": epoch_total_loss,
        "reconstruction_loss": epoch_reconstruction_loss,
        "sparsity_loss": epoch_sparsity_loss,
    })

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"total_loss={epoch_total_loss:.6f} | "
        f"reconstruction_loss={epoch_reconstruction_loss:.6f} | "
        f"sparsity_loss={epoch_sparsity_loss:.6f}"
    )

checkpoint_dir = project_root / "outputs" / "sae_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "sae_layer06_diverse200.pt"
metrics_path = checkpoint_dir / "sae_layer06_diverse200_metrics.csv"

torch.save({
    "model_state_dict": sae.state_dict(),
    "d_in": d_in,
    "d_sae": d_sae,
    "l1_coeff": l1_coeff,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "learning_rate": learning_rate,
}, checkpoint_path)

pd.DataFrame(metrics).to_csv(metrics_path, index=False)

print(f"Saved checkpoint path: {checkpoint_path}")
print(f"Saved metrics path: {metrics_path}")

# Rank and visualize features from diverse-200 SAE

We select features with moderate activation frequency to avoid dead features and overly broad features. For each selected feature, cropped top patches show the local visual pattern, while full-image context shows where the feature activates in the original image.

In [ ]:
from PIL import ImageDraw

activation_path = project_root / "outputs" / "activations" / "layer_06_patch_tokens_diverse_200.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_06_metadata_diverse_200.csv"
checkpoint_path = project_root / "outputs" / "sae_checkpoints" / "sae_layer06_diverse200.pt"

patch_tokens = torch.load(activation_path, map_location="cpu")
acts = patch_tokens.reshape(-1, 768).float()
metadata = pd.read_csv(metadata_path)

assert len(metadata) == acts.shape[0]

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=4096):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

d_in = 768
d_sae = 4096
sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

with torch.no_grad():
    z = F.relu(sae.encoder(acts.to(device))).cpu()

print(f"acts shape: {acts.shape}")
print(f"z shape: {z.shape}")
assert list(z.shape) == [39200, 4096]

mean_activation = z.mean(dim=0)
max_activation = z.max(dim=0).values
activation_frequency = (z > 0).float().mean(dim=0)

feature_stats = pd.DataFrame({
    "feature_id": range(d_sae),
    "mean_activation": mean_activation.numpy(),
    "max_activation": max_activation.numpy(),
    "activation_frequency": activation_frequency.numpy(),
})

ranking_dir = project_root / "outputs" / "feature_rankings"
visualization_dir = project_root / "outputs" / "feature_visualizations"
ranking_dir.mkdir(parents=True, exist_ok=True)
visualization_dir.mkdir(parents=True, exist_ok=True)

feature_stats_path = ranking_dir / "layer06_diverse200_feature_stats.csv"
feature_stats.to_csv(feature_stats_path, index=False)

candidate_features = feature_stats[
    (feature_stats["activation_frequency"] >= 0.01)
    & (feature_stats["activation_frequency"] <= 0.20)
].sort_values("max_activation", ascending=False).head(10)

selected_feature_ids = candidate_features["feature_id"].tolist()
print("Selected feature ids:", selected_feature_ids)
print(candidate_features)

top_patch_rows = []

for feature_id in selected_feature_ids:
    feature_activations = z[:, feature_id]
    top_values, top_indices = torch.topk(feature_activations, k=10)

    feature_top_rows = metadata.iloc[top_indices.numpy()].copy()
    feature_top_rows.insert(0, "feature_id", feature_id)
    feature_top_rows.insert(1, "rank", range(1, 11))
    feature_top_rows["feature_activation"] = top_values.numpy()
    top_patch_rows.append(feature_top_rows)

top_patches = pd.concat(top_patch_rows, ignore_index=True)
top_patches_path = ranking_dir / "layer06_diverse200_top_patches.csv"
top_patches.to_csv(top_patches_path, index=False)

print(f"Saved feature stats path: {feature_stats_path}")
print(f"Saved top patches path: {top_patches_path}")

for feature_id in selected_feature_ids:
    feature_rows = top_patches[top_patches["feature_id"] == feature_id].sort_values(
        "feature_activation", ascending=False
    ).head(10)

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle(f"Top activating patches for SAE feature {feature_id}")

    for ax, (_, row) in zip(axes.flat, feature_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])
        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16
        patch = img.crop((left, upper, right, lower))

        ax.imshow(patch)
        ax.set_title(f"{row['feature_activation']:.3f}\nr{patch_row}, c{patch_col}")
        ax.axis("off")

    plt.tight_layout()
    crop_vis_path = visualization_dir / f"layer06_diverse200_feature_{feature_id}_top_patches.png"
    plt.savefig(crop_vis_path, dpi=150)
    plt.show()

    context_rows = feature_rows.head(5)
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f"Full image context for SAE feature {feature_id}")

    for ax, (_, row) in zip(axes.flat, context_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])
        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16

        draw = ImageDraw.Draw(img)
        draw.rectangle((left, upper, right, lower), outline="red", width=3)

        ax.imshow(img)
        ax.set_title(
            f"f{feature_id} | {row['feature_activation']:.3f}\n"
            f"r{patch_row}, c{patch_col}\n{row['class_name']}"
        )
        ax.axis("off")

    plt.tight_layout()
    context_vis_path = visualization_dir / f"layer06_diverse200_feature_{feature_id}_full_image_context.png"
    plt.savefig(context_vis_path, dpi=150)
    plt.show()

    print(f"Saved cropped-patch visualization: {crop_vis_path}")
    print(f"Saved full-image context visualization: {context_vis_path}")

# Create feature interpretation notes

These notes are manually written after inspecting top activating patches and full-image context. They will be used later for human evaluation and layer-level interpretation analysis.

In [ ]:
import pandas as pd
from pathlib import Path

interpretation_notes = pd.DataFrame([
    {
        "feature_id": 245,
        "possible_label": "pale smooth background",
        "visual_pattern": "light gray, light blue, light green, low-texture patches",
        "object_or_background": "background",
        "semantic_level": "low-level visual/background feature",
        "confidence": "medium",
        "evidence_summary": "Top activating patches are mostly smooth pale regions. Full-image context shows red boxes mostly on background areas rather than object bodies.",
        "notes": "This feature does not appear to correspond to a specific object class or object part.",
    }
], columns=[
    "feature_id",
    "possible_label",
    "visual_pattern",
    "object_or_background",
    "semantic_level",
    "confidence",
    "evidence_summary",
    "notes",
])

notes_path = project_root / "outputs" / "feature_rankings" / "layer06_diverse200_interpretation_notes.csv"
notes_path.parent.mkdir(parents=True, exist_ok=True)
interpretation_notes.to_csv(notes_path, index=False)

print(f"Saved interpretation notes path: {notes_path}")
print(interpretation_notes)

# Generate full-image context grids for candidate features

Full-image context helps us determine whether a feature activates on the object, object part, background, texture, or image boundary. This is more informative than cropped 16x16 patches alone.

In [ ]:
feature_stats_path = project_root / "outputs" / "feature_rankings" / "layer06_diverse200_feature_stats.csv"
top_patches_path = project_root / "outputs" / "feature_rankings" / "layer06_diverse200_top_patches.csv"
interpretation_notes_path = project_root / "outputs" / "feature_rankings" / "layer06_diverse200_interpretation_notes.csv"

feature_stats = pd.read_csv(feature_stats_path)
top_patches = pd.read_csv(top_patches_path)
interpretation_notes = pd.read_csv(interpretation_notes_path)

available_feature_ids = top_patches["feature_id"].drop_duplicates().tolist()
recorded_feature_ids = set(interpretation_notes["feature_id"].dropna().astype(int).tolist())
candidate_feature_ids = [
    int(feature_id)
    for feature_id in available_feature_ids
    if int(feature_id) not in recorded_feature_ids
][:10]

print("Candidate feature ids:", candidate_feature_ids)

visualization_dir = project_root / "outputs" / "feature_visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
saved_paths = []

for feature_id in candidate_feature_ids:
    feature_rows = top_patches[top_patches["feature_id"] == feature_id].sort_values(
        "feature_activation", ascending=False
    ).head(5)

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f"Full image context for SAE feature {feature_id}")

    for ax, (_, row) in zip(axes.flat, feature_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])
        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16

        draw = ImageDraw.Draw(img)
        draw.rectangle((left, upper, right, lower), outline="red", width=3)

        ax.imshow(img)
        ax.set_title(
            f"f{feature_id} | {row['feature_activation']:.3f}\n"
            f"r{patch_row}, c{patch_col}\n{row['class_name']}"
        )
        ax.axis("off")

    for ax in axes.flat[len(feature_rows):]:
        ax.axis("off")

    plt.tight_layout()
    context_vis_path = visualization_dir / f"layer06_diverse200_feature_{feature_id}_full_image_context.png"
    plt.savefig(context_vis_path, dpi=150)
    plt.show()

    saved_paths.append(context_vis_path)

print("Saved full-image context grids:")
for path in saved_paths:
    print(path)

# Extract layer 12 activations for diverse subset 200

We extract layer 12 activations to compare whether later ViT layers produce SAE features that are more object-related than layer 6 features.

In [ ]:
subset_path = project_root / "outputs" / "activations" / "diverse_subset_200.csv"
diverse_subset_200 = pd.read_csv(subset_path)

image_paths_200 = diverse_subset_200["image_path"].tolist()
batch_size = 32

target_layer = model.encoder.layers[11]
batch_activations = {}

def save_layer_12_batch_activation(module, inputs, output):
    batch_activations["layer_12"] = output.detach().cpu()

hook_handle = target_layer.register_forward_hook(save_layer_12_batch_activation)
patch_token_batches = []

for start_idx in range(0, len(image_paths_200), batch_size):
    end_idx = min(start_idx + batch_size, len(image_paths_200))
    batch_paths = image_paths_200[start_idx:end_idx]

    batch_images = []
    for path in batch_paths:
        img = Image.open(path).convert("RGB")
        batch_images.append(preprocess(img))

    batch_x = torch.stack(batch_images, dim=0).to(device)

    with torch.no_grad():
        _ = model(batch_x)

    activation = batch_activations["layer_12"]
    patch_tokens_batch = activation[:, 1:, :].cpu()
    patch_token_batches.append(patch_tokens_batch)

    print(f"Processed images {start_idx} to {end_idx - 1}: patch_tokens shape {patch_tokens_batch.shape}")

hook_handle.remove()

patch_tokens = torch.cat(patch_token_batches, dim=0)

metadata_rows = []
for _, image_row in diverse_subset_200.iterrows():
    for patch_id in range(196):
        metadata_rows.append({
            "image_index": int(image_row["image_index"]),
            "image_path": image_row["image_path"],
            "class_id": image_row["class_id"],
            "class_name": image_row["class_name"],
            "patch_id": patch_id,
            "patch_row": patch_id // 14,
            "patch_col": patch_id % 14,
        })

metadata = pd.DataFrame(metadata_rows)

activation_path = project_root / "outputs" / "activations" / "layer_12_patch_tokens_diverse_200.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_12_metadata_diverse_200.csv"

torch.save(patch_tokens, activation_path)
metadata.to_csv(metadata_path, index=False)

print(f"Number of images processed: {len(image_paths_200)}")
print(f"Final patch_tokens shape: {patch_tokens.shape}")
print(f"Metadata rows: {len(metadata)}")
print(f"Saved activation path: {activation_path}")
print(f"Saved metadata path: {metadata_path}")

assert list(patch_tokens.shape) == [200, 196, 768]
assert len(metadata) == 200 * 196

# Train SAE on layer 12 diverse subset 200

We train a separate SAE for layer 12 because different ViT layers have different activation distributions and may encode different levels of visual abstraction. Using the same settings as layer 6 makes the comparison more controlled.

In [ ]:
activation_path = project_root / "outputs" / "activations" / "layer_12_patch_tokens_diverse_200.pt"
patch_tokens = torch.load(activation_path, map_location="cpu")
acts = patch_tokens.reshape(-1, 768)

print(f"Original activation shape: {patch_tokens.shape}")
print(f"Flattened activation shape: {acts.shape}")

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=4096):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

d_in = 768
d_sae = 4096
l1_coeff = 3e-3
batch_size = 1024
num_epochs = 30
learning_rate = 1e-3

sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)
dataset = TensorDataset(acts.float())
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
optimizer = torch.optim.Adam(sae.parameters(), lr=learning_rate)

metrics = []

for epoch in range(num_epochs):
    sae.train()
    total_loss_sum = 0.0
    reconstruction_loss_sum = 0.0
    sparsity_loss_sum = 0.0
    num_examples = 0

    for (batch_acts,) in dataloader:
        batch_acts = batch_acts.to(device)

        optimizer.zero_grad()
        x_hat, z = sae(batch_acts)
        reconstruction_loss = F.mse_loss(x_hat, batch_acts)
        sparsity_loss = z.abs().mean()
        total_loss = reconstruction_loss + l1_coeff * sparsity_loss
        total_loss.backward()
        optimizer.step()

        current_batch_size = batch_acts.shape[0]
        total_loss_sum += total_loss.item() * current_batch_size
        reconstruction_loss_sum += reconstruction_loss.item() * current_batch_size
        sparsity_loss_sum += sparsity_loss.item() * current_batch_size
        num_examples += current_batch_size

    epoch_total_loss = total_loss_sum / num_examples
    epoch_reconstruction_loss = reconstruction_loss_sum / num_examples
    epoch_sparsity_loss = sparsity_loss_sum / num_examples

    metrics.append({
        "epoch": epoch + 1,
        "total_loss": epoch_total_loss,
        "reconstruction_loss": epoch_reconstruction_loss,
        "sparsity_loss": epoch_sparsity_loss,
    })

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"total_loss={epoch_total_loss:.6f} | "
        f"reconstruction_loss={epoch_reconstruction_loss:.6f} | "
        f"sparsity_loss={epoch_sparsity_loss:.6f}"
    )

checkpoint_dir = project_root / "outputs" / "sae_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "sae_layer12_diverse200.pt"
metrics_path = checkpoint_dir / "sae_layer12_diverse200_metrics.csv"

torch.save({
    "model_state_dict": sae.state_dict(),
    "d_in": d_in,
    "d_sae": d_sae,
    "l1_coeff": l1_coeff,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "learning_rate": learning_rate,
}, checkpoint_path)

pd.DataFrame(metrics).to_csv(metrics_path, index=False)

print(f"Saved checkpoint path: {checkpoint_path}")
print(f"Saved metrics path: {metrics_path}")

# Rank and visualize features from layer 12 SAE

We inspect layer 12 SAE features to test whether later ViT layers produce more object-related or semantic features than layer 6. Cropped patches show local patterns, while full-image context shows whether activations fall on objects, object parts, or background.

In [ ]:
activation_path = project_root / "outputs" / "activations" / "layer_12_patch_tokens_diverse_200.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_12_metadata_diverse_200.csv"
checkpoint_path = project_root / "outputs" / "sae_checkpoints" / "sae_layer12_diverse200.pt"

patch_tokens = torch.load(activation_path, map_location="cpu")
acts = patch_tokens.reshape(-1, 768).float()
metadata = pd.read_csv(metadata_path)

assert len(metadata) == acts.shape[0]

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=4096):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

d_in = 768
d_sae = 4096
sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

with torch.no_grad():
    z = F.relu(sae.encoder(acts.to(device))).cpu()

print(f"acts shape: {acts.shape}")
print(f"z shape: {z.shape}")
assert list(z.shape) == [39200, 4096]

mean_activation = z.mean(dim=0)
max_activation = z.max(dim=0).values
activation_frequency = (z > 0).float().mean(dim=0)

feature_stats = pd.DataFrame({
    "feature_id": range(d_sae),
    "mean_activation": mean_activation.numpy(),
    "max_activation": max_activation.numpy(),
    "activation_frequency": activation_frequency.numpy(),
})

ranking_dir = project_root / "outputs" / "feature_rankings"
visualization_dir = project_root / "outputs" / "feature_visualizations"
ranking_dir.mkdir(parents=True, exist_ok=True)
visualization_dir.mkdir(parents=True, exist_ok=True)

feature_stats_path = ranking_dir / "layer12_diverse200_feature_stats.csv"
feature_stats.to_csv(feature_stats_path, index=False)

candidate_features = feature_stats[
    (feature_stats["activation_frequency"] >= 0.01)
    & (feature_stats["activation_frequency"] <= 0.20)
].sort_values("max_activation", ascending=False).head(10)

selected_feature_ids = candidate_features["feature_id"].tolist()
print("Selected feature ids:", selected_feature_ids)
print(candidate_features)

top_patch_rows = []

for feature_id in selected_feature_ids:
    feature_activations = z[:, feature_id]
    top_values, top_indices = torch.topk(feature_activations, k=10)

    feature_top_rows = metadata.iloc[top_indices.numpy()].copy()
    feature_top_rows.insert(0, "feature_id", feature_id)
    feature_top_rows.insert(1, "rank", range(1, 11))
    feature_top_rows["feature_activation"] = top_values.numpy()
    top_patch_rows.append(feature_top_rows)

top_patches = pd.concat(top_patch_rows, ignore_index=True)
top_patches_path = ranking_dir / "layer12_diverse200_top_patches.csv"
top_patches.to_csv(top_patches_path, index=False)

print(f"Saved feature stats path: {feature_stats_path}")
print(f"Saved top patches path: {top_patches_path}")

for feature_id in selected_feature_ids:
    feature_rows = top_patches[top_patches["feature_id"] == feature_id].sort_values(
        "feature_activation", ascending=False
    ).head(10)

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle(f"Top activating patches for layer 12 SAE feature {feature_id}")

    for ax, (_, row) in zip(axes.flat, feature_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])
        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16
        patch = img.crop((left, upper, right, lower))

        ax.imshow(patch)
        ax.set_title(f"{row['feature_activation']:.3f}\nr{patch_row}, c{patch_col}")
        ax.axis("off")

    plt.tight_layout()
    crop_vis_path = visualization_dir / f"layer12_diverse200_feature_{feature_id}_top_patches.png"
    plt.savefig(crop_vis_path, dpi=150)
    plt.show()

    context_rows = feature_rows.head(5)
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f"Full image context for layer 12 SAE feature {feature_id}")

    for ax, (_, row) in zip(axes.flat, context_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])
        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16

        draw = ImageDraw.Draw(img)
        draw.rectangle((left, upper, right, lower), outline="red", width=3)

        ax.imshow(img)
        ax.set_title(
            f"f{feature_id} | {row['feature_activation']:.3f}\n"
            f"r{patch_row}, c{patch_col}\n{row['class_name']}"
        )
        ax.axis("off")

    plt.tight_layout()
    context_vis_path = visualization_dir / f"layer12_diverse200_feature_{feature_id}_full_image_context.png"
    plt.savefig(context_vis_path, dpi=150)
    plt.show()

    print(f"Saved cropped-patch visualization: {crop_vis_path}")
    print(f"Saved full-image context visualization: {context_vis_path}")

# Rank SAE features using center-object patch filtering

Because unrestricted top-activation ranking mostly selected background patches, we now restrict ranking to center-region patches as a simple proxy for object-related regions. This is not a perfect object mask, but it helps test whether SAE features become more object-related when background patches are reduced.

In [ ]:
activation_path = project_root / "outputs" / "activations" / "layer_12_patch_tokens_diverse_200.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_12_metadata_diverse_200.csv"
checkpoint_path = project_root / "outputs" / "sae_checkpoints" / "sae_layer12_diverse200.pt"

patch_tokens = torch.load(activation_path, map_location="cpu")
acts = patch_tokens.reshape(-1, 768).float()
metadata = pd.read_csv(metadata_path)

assert len(metadata) == acts.shape[0]

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=4096):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

d_in = 768
d_sae = 4096
sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

with torch.no_grad():
    z = F.relu(sae.encoder(acts.to(device))).cpu()

center_mask = metadata["patch_row"].between(4, 9) & metadata["patch_col"].between(4, 9)
center_mask_values = center_mask.to_numpy()
center_z = z[center_mask_values]
center_metadata = metadata[center_mask].reset_index(drop=True)

print(f"Total patches: {len(metadata)}")
print(f"Center patches: {len(center_metadata)}")
print(f"center_z shape: {center_z.shape}")

mean_activation = center_z.mean(dim=0)
max_activation = center_z.max(dim=0).values
activation_frequency = (center_z > 0).float().mean(dim=0)

center_feature_stats = pd.DataFrame({
    "feature_id": range(d_sae),
    "mean_activation": mean_activation.numpy(),
    "max_activation": max_activation.numpy(),
    "activation_frequency": activation_frequency.numpy(),
})

candidate_features = center_feature_stats[
    (center_feature_stats["activation_frequency"] >= 0.01)
    & (center_feature_stats["activation_frequency"] <= 0.20)
].sort_values("max_activation", ascending=False).head(10)

selected_feature_ids = candidate_features["feature_id"].tolist()
print("Selected center-filtered feature ids:", selected_feature_ids)
print(candidate_features)

ranking_dir = project_root / "outputs" / "feature_rankings"
visualization_dir = project_root / "outputs" / "feature_visualizations"
ranking_dir.mkdir(parents=True, exist_ok=True)
visualization_dir.mkdir(parents=True, exist_ok=True)

top_patch_rows = []

for feature_id in selected_feature_ids:
    feature_activations = center_z[:, feature_id]
    top_values, top_indices = torch.topk(feature_activations, k=10)

    feature_top_rows = center_metadata.iloc[top_indices.numpy()].copy()
    feature_top_rows.insert(0, "feature_id", feature_id)
    feature_top_rows.insert(1, "rank", range(1, 11))
    feature_top_rows["feature_activation"] = top_values.numpy()
    top_patch_rows.append(feature_top_rows)

top_patches = pd.concat(top_patch_rows, ignore_index=True)
top_patches_path = ranking_dir / "layer12_center_filtered_top_patches.csv"
top_patches.to_csv(top_patches_path, index=False)

print(f"Saved center-filtered top patches path: {top_patches_path}")

for feature_id in selected_feature_ids:
    feature_rows = top_patches[top_patches["feature_id"] == feature_id].sort_values(
        "feature_activation", ascending=False
    ).head(10)

    context_rows = feature_rows.head(5)
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f"Center-filtered full image context for layer 12 SAE feature {feature_id}")

    for ax, (_, row) in zip(axes.flat, context_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])
        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16

        draw = ImageDraw.Draw(img)
        draw.rectangle((left, upper, right, lower), outline="red", width=3)

        ax.imshow(img)
        ax.set_title(
            f"f{feature_id} | {row['feature_activation']:.3f}\n"
            f"r{patch_row}, c{patch_col}\n{row['class_name']}"
        )
        ax.axis("off")

    plt.tight_layout()
    context_vis_path = visualization_dir / f"layer12_center_feature_{feature_id}_full_image_context.png"
    plt.savefig(context_vis_path, dpi=150)
    plt.show()

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle(f"Center-filtered top patches for layer 12 SAE feature {feature_id}")

    for ax, (_, row) in zip(axes.flat, feature_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        patch_row = int(row["patch_row"])
        patch_col = int(row["patch_col"])
        left = patch_col * 16
        upper = patch_row * 16
        right = left + 16
        lower = upper + 16
        patch = img.crop((left, upper, right, lower))

        ax.imshow(patch)
        ax.set_title(f"{row['feature_activation']:.3f}\nr{patch_row}, c{patch_col}")
        ax.axis("off")

    plt.tight_layout()
    crop_vis_path = visualization_dir / f"layer12_center_feature_{feature_id}_top_patches.png"
    plt.savefig(crop_vis_path, dpi=150)
    plt.show()

    print(f"Saved full-image context visualization: {context_vis_path}")
    print(f"Saved cropped-patch visualization: {crop_vis_path}")

# Extract layer 12 CLS token activations for diverse subset 200

Patch-token features are local and often dominated by background or texture. CLS-token activations aggregate information across the image, so SAE features trained on CLS tokens may better reflect image-level or category-related concepts.

In [ ]:
subset_path = project_root / "outputs" / "activations" / "diverse_subset_200.csv"
diverse_subset_200 = pd.read_csv(subset_path)

image_paths_200 = diverse_subset_200["image_path"].tolist()
batch_size = 32

target_layer = model.encoder.layers[11]
batch_activations = {}

def save_layer_12_cls_batch_activation(module, inputs, output):
    batch_activations["layer_12"] = output.detach().cpu()

hook_handle = target_layer.register_forward_hook(save_layer_12_cls_batch_activation)
cls_token_batches = []

for start_idx in range(0, len(image_paths_200), batch_size):
    end_idx = min(start_idx + batch_size, len(image_paths_200))
    batch_paths = image_paths_200[start_idx:end_idx]

    batch_images = []
    for path in batch_paths:
        img = Image.open(path).convert("RGB")
        batch_images.append(preprocess(img))

    batch_x = torch.stack(batch_images, dim=0).to(device)

    with torch.no_grad():
        _ = model(batch_x)

    activation = batch_activations["layer_12"]
    cls_tokens_batch = activation[:, 0, :].cpu()
    cls_token_batches.append(cls_tokens_batch)

    print(f"Processed images {start_idx} to {end_idx - 1}: CLS tokens shape {cls_tokens_batch.shape}")

hook_handle.remove()

cls_tokens = torch.cat(cls_token_batches, dim=0)
metadata = diverse_subset_200[["image_index", "image_path", "class_id", "class_name"]].copy()

activation_path = project_root / "outputs" / "activations" / "layer_12_cls_tokens_diverse_200.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_12_cls_metadata_diverse_200.csv"

torch.save(cls_tokens, activation_path)
metadata.to_csv(metadata_path, index=False)

print(f"Number of images processed: {len(image_paths_200)}")
print(f"Final CLS token shape: {cls_tokens.shape}")
print(f"Metadata rows: {len(metadata)}")
print(f"Saved activation path: {activation_path}")
print(f"Saved metadata path: {metadata_path}")

assert list(cls_tokens.shape) == [200, 768]
assert len(metadata) == 200

# Train SAE on layer 12 CLS tokens

Unlike patch-token SAE features, CLS-token SAE features are image-level. After training, we will interpret each SAE feature by retrieving the top activating full images rather than image patches.

In [ ]:
activation_path = project_root / "outputs" / "activations" / "layer_12_cls_tokens_diverse_200.pt"
cls_tokens = torch.load(activation_path, map_location="cpu")

print(f"CLS token activation shape: {cls_tokens.shape}")

acts = cls_tokens.float()

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=1024):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

d_in = 768
d_sae = 1024
l1_coeff = 3e-3
batch_size = 64
num_epochs = 50
learning_rate = 1e-3

sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)
dataset = TensorDataset(acts)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
optimizer = torch.optim.Adam(sae.parameters(), lr=learning_rate)

metrics = []

for epoch in range(num_epochs):
    sae.train()
    total_loss_sum = 0.0
    reconstruction_loss_sum = 0.0
    sparsity_loss_sum = 0.0
    num_examples = 0

    for (batch_acts,) in dataloader:
        batch_acts = batch_acts.to(device)

        optimizer.zero_grad()
        x_hat, z = sae(batch_acts)
        reconstruction_loss = F.mse_loss(x_hat, batch_acts)
        sparsity_loss = z.abs().mean()
        total_loss = reconstruction_loss + l1_coeff * sparsity_loss
        total_loss.backward()
        optimizer.step()

        current_batch_size = batch_acts.shape[0]
        total_loss_sum += total_loss.item() * current_batch_size
        reconstruction_loss_sum += reconstruction_loss.item() * current_batch_size
        sparsity_loss_sum += sparsity_loss.item() * current_batch_size
        num_examples += current_batch_size

    epoch_total_loss = total_loss_sum / num_examples
    epoch_reconstruction_loss = reconstruction_loss_sum / num_examples
    epoch_sparsity_loss = sparsity_loss_sum / num_examples

    metrics.append({
        "epoch": epoch + 1,
        "total_loss": epoch_total_loss,
        "reconstruction_loss": epoch_reconstruction_loss,
        "sparsity_loss": epoch_sparsity_loss,
    })

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"total_loss={epoch_total_loss:.6f} | "
        f"reconstruction_loss={epoch_reconstruction_loss:.6f} | "
        f"sparsity_loss={epoch_sparsity_loss:.6f}"
    )

checkpoint_dir = project_root / "outputs" / "sae_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "sae_layer12_cls_diverse200.pt"
metrics_path = checkpoint_dir / "sae_layer12_cls_diverse200_metrics.csv"

torch.save({
    "model_state_dict": sae.state_dict(),
    "d_in": d_in,
    "d_sae": d_sae,
    "l1_coeff": l1_coeff,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "learning_rate": learning_rate,
}, checkpoint_path)

pd.DataFrame(metrics).to_csv(metrics_path, index=False)

print(f"Saved checkpoint path: {checkpoint_path}")
print(f"Saved metrics path: {metrics_path}")

# Rank and visualize top activating images for CLS-token SAE

CLS-token SAE features are interpreted by their top activating images. If the top images share similar classes, objects, or scenes, the feature may represent an image-level semantic direction.

In [ ]:
activation_path = project_root / "outputs" / "activations" / "layer_12_cls_tokens_diverse_200.pt"
metadata_path = project_root / "outputs" / "activations" / "layer_12_cls_metadata_diverse_200.csv"
checkpoint_path = project_root / "outputs" / "sae_checkpoints" / "sae_layer12_cls_diverse200.pt"

cls_tokens = torch.load(activation_path, map_location="cpu").float()
metadata = pd.read_csv(metadata_path)

assert len(metadata) == cls_tokens.shape[0]

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=1024):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

d_in = 768
d_sae = 1024
sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

with torch.no_grad():
    z = F.relu(sae.encoder(cls_tokens.to(device))).cpu()

print(f"cls_tokens shape: {cls_tokens.shape}")
print(f"z shape: {z.shape}")
assert list(z.shape) == [200, 1024]

mean_activation = z.mean(dim=0)
max_activation = z.max(dim=0).values
activation_frequency = (z > 0).float().mean(dim=0)

feature_stats = pd.DataFrame({
    "feature_id": range(d_sae),
    "mean_activation": mean_activation.numpy(),
    "max_activation": max_activation.numpy(),
    "activation_frequency": activation_frequency.numpy(),
})

ranking_dir = project_root / "outputs" / "feature_rankings"
visualization_dir = project_root / "outputs" / "feature_visualizations"
ranking_dir.mkdir(parents=True, exist_ok=True)
visualization_dir.mkdir(parents=True, exist_ok=True)

feature_stats_path = ranking_dir / "layer12_cls_diverse200_feature_stats.csv"
feature_stats.to_csv(feature_stats_path, index=False)

candidate_features = feature_stats[
    (feature_stats["activation_frequency"] >= 0.05)
    & (feature_stats["activation_frequency"] <= 0.50)
].sort_values("max_activation", ascending=False).head(10)

selected_feature_ids = candidate_features["feature_id"].tolist()
print("Selected CLS feature ids:", selected_feature_ids)
print(candidate_features)

top_image_rows = []

for feature_id in selected_feature_ids:
    feature_activations = z[:, feature_id]
    top_values, top_indices = torch.topk(feature_activations, k=10)

    feature_top_rows = metadata.iloc[top_indices.numpy()].copy()
    feature_top_rows.insert(0, "feature_id", feature_id)
    feature_top_rows.insert(1, "rank", range(1, 11))
    feature_top_rows["feature_activation"] = top_values.numpy()
    top_image_rows.append(feature_top_rows)

top_images = pd.concat(top_image_rows, ignore_index=True)
top_images_path = ranking_dir / "layer12_cls_diverse200_top_images.csv"
top_images.to_csv(top_images_path, index=False)

print(f"Saved feature stats path: {feature_stats_path}")
print(f"Saved top images path: {top_images_path}")

for feature_id in selected_feature_ids:
    feature_rows = top_images[top_images["feature_id"] == feature_id].sort_values(
        "feature_activation", ascending=False
    ).head(10)

    fig, axes = plt.subplots(2, 5, figsize=(14, 7))
    fig.suptitle(f"Top activating images for layer 12 CLS SAE feature {feature_id}")

    for ax, (_, row) in zip(axes.flat, feature_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        ax.imshow(img)
        ax.set_title(f"{row['feature_activation']:.3f}\n{row['class_name']}")
        ax.axis("off")

    plt.tight_layout()
    vis_path = visualization_dir / f"layer12_cls_feature_{feature_id}_top_images.png"
    plt.savefig(vis_path, dpi=150)
    plt.show()

    print(f"Saved top-image visualization: {vis_path}")

# Compute top-image class purity for CLS SAE features

Top-10 class purity measures whether a CLS SAE feature is category-related. A high purity means the feature's top activating images mostly come from the same class, suggesting a more semantic or class-specific direction.

In [ ]:
top_images_path = project_root / "outputs" / "feature_rankings" / "layer12_cls_diverse200_top_images.csv"
top_images = pd.read_csv(top_images_path)

print("Columns:", top_images.columns.tolist())
print(top_images.head())

purity_rows = []

for feature_id, feature_rows in top_images.groupby("feature_id"):
    top10 = feature_rows.sort_values("feature_activation", ascending=False).head(10)
    class_counts = top10["class_name"].value_counts()

    dominant_class_name = class_counts.index[0]
    dominant_class_count = int(class_counts.iloc[0])
    top10_purity = dominant_class_count / 10
    class_distribution = "; ".join(
        f"{class_name}: {count}"
        for class_name, count in class_counts.items()
    )

    purity_rows.append({
        "feature_id": int(feature_id),
        "dominant_class_name": dominant_class_name,
        "dominant_class_count": dominant_class_count,
        "top10_purity": top10_purity,
        "class_distribution": class_distribution,
        "mean_top10_activation": top10["feature_activation"].mean(),
        "max_activation": top10["feature_activation"].max(),
    })

class_purity = pd.DataFrame(purity_rows).sort_values(
    ["top10_purity", "max_activation"], ascending=[False, False]
).reset_index(drop=True)

purity_path = project_root / "outputs" / "feature_rankings" / "layer12_cls_diverse200_top10_class_purity.csv"
class_purity.to_csv(purity_path, index=False)

print(f"Saved class purity path: {purity_path}")
print(class_purity)

# Visualize high-purity CLS SAE features

High-purity CLS SAE features are features whose top activating images mostly belong to the same class. These are candidate category-related features and can be manually inspected for semantic interpretation.

In [ ]:
top_images_path = project_root / "outputs" / "feature_rankings" / "layer12_cls_diverse200_top_images.csv"
purity_path = project_root / "outputs" / "feature_rankings" / "layer12_cls_diverse200_top10_class_purity.csv"

top_images = pd.read_csv(top_images_path)
class_purity = pd.read_csv(purity_path)

selected_features = class_purity[class_purity["top10_purity"] >= 0.7].sort_values(
    ["top10_purity", "max_activation"], ascending=[False, False]
).head(10)

print(selected_features[[
    "feature_id",
    "dominant_class_name",
    "dominant_class_count",
    "top10_purity",
    "class_distribution",
]])

visualization_dir = project_root / "outputs" / "feature_visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []

for _, feature_info in selected_features.iterrows():
    feature_id = int(feature_info["feature_id"])
    feature_rows = top_images[top_images["feature_id"] == feature_id].sort_values(
        "feature_activation", ascending=False
    ).head(10)

    fig, axes = plt.subplots(2, 5, figsize=(14, 7))
    fig.suptitle(
        f"High-purity CLS SAE feature {feature_id}: {feature_info['dominant_class_name']} "
        f"(purity={feature_info['top10_purity']:.2f})"
    )

    for ax, (_, row) in zip(axes.flat, feature_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize((224, 224))
        ax.imshow(img)
        ax.set_title(f"{row['feature_activation']:.3f}\n{row['class_name']}")
        ax.axis("off")

    plt.tight_layout()
    vis_path = visualization_dir / f"layer12_cls_high_purity_feature_{feature_id}_top_images.png"
    plt.savefig(vis_path, dpi=150)
    plt.show()

    summary_rows.append({
        "feature_id": feature_id,
        "dominant_class_name": feature_info["dominant_class_name"],
        "top10_purity": feature_info["top10_purity"],
        "class_distribution": feature_info["class_distribution"],
        "visualization_path": str(vis_path),
    })

    print(f"Saved high-purity visualization: {vis_path}")

summary = pd.DataFrame(summary_rows, columns=[
    "feature_id",
    "dominant_class_name",
    "top10_purity",
    "class_distribution",
    "visualization_path",
])

summary_path = project_root / "outputs" / "feature_rankings" / "layer12_cls_high_purity_features_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"Saved high-purity summary path: {summary_path}")
print(summary)

# Add CLS feature interpretation notes

This notes file records manual interpretations of CLS-token SAE features. Unlike patch-token features, CLS-token features are interpreted by top activating full images and may reveal category-level concepts.

In [ ]:
notes_path = project_root / "outputs" / "feature_rankings" / "layer12_cls_diverse200_interpretation_notes.csv"

note_columns = [
    "feature_id",
    "possible_label",
    "visual_pattern",
    "object_or_background",
    "semantic_level",
    "confidence",
    "top10_purity",
    "evidence_summary",
    "notes",
]

if notes_path.exists():
    interpretation_notes = pd.read_csv(notes_path)
else:
    interpretation_notes = pd.DataFrame(columns=note_columns)

new_note = {
    "feature_id": 439,
    "possible_label": "electric ray / ray-like aquatic animal",
    "visual_pattern": "flat aquatic animal body, underwater or aquarium scenes",
    "object_or_background": "image-level object/category",
    "semantic_level": "category-related",
    "confidence": "medium-high",
    "top10_purity": 0.8,
    "evidence_summary": "The top 10 activating images are dominated by electric ray images, with 8 out of 10 images from the electric ray class.",
    "notes": "This feature is not perfectly class-specific because a few coucal images also activate it, but it is much more semantically coherent than the patch-token SAE features.",
}

interpretation_notes = interpretation_notes[interpretation_notes["feature_id"] != new_note["feature_id"]]
interpretation_notes = pd.concat([
    interpretation_notes,
    pd.DataFrame([new_note], columns=note_columns),
], ignore_index=True)

interpretation_notes = interpretation_notes[note_columns].sort_values("feature_id").reset_index(drop=True)
notes_path.parent.mkdir(parents=True, exist_ok=True)
interpretation_notes.to_csv(notes_path, index=False)

print(f"Saved CLS interpretation notes path: {notes_path}")
print(interpretation_notes)

# Summarize current SAE experiment results

This summary table records the current experimental findings. It helps compare patch-token and CLS-token SAE analysis and clarifies which results are reliable, which are exploratory, and what should be done next.

In [ ]:
import pandas as pd
from pathlib import Path

experiment_summary = pd.DataFrame([
    {
        "experiment_id": "layer6_patch_sae",
        "representation_type": "patch token",
        "layer": "6",
        "activation_shape": "[200, 196, 768]",
        "sae_input_unit": "one 768-d patch-token activation",
        "interpretation_unit": "top activating image patches",
        "selection_method": "top feature activations, full-image context inspection",
        "main_observation": "Most inspected features activate on background, color, water, smooth regions, or texture patches.",
        "example_feature": "feature 245",
        "evidence": "Feature 245 activates on pale smooth background regions rather than object bodies.",
        "limitation": "Patch-token features are local and may be dominated by background patches.",
        "next_step": "Compare with deeper layers and CLS-token representations.",
    },
    {
        "experiment_id": "layer12_patch_sae",
        "representation_type": "patch token",
        "layer": "12",
        "activation_shape": "[200, 196, 768]",
        "sae_input_unit": "one 768-d patch-token activation",
        "interpretation_unit": "top activating image patches",
        "selection_method": "top feature activations plus center-region patch filtering",
        "main_observation": "Even in the final ViT layer, inspected patch-token features mostly activate on background, color, or texture regions.",
        "example_feature": "N/A",
        "evidence": "Full-image context visualizations show red boxes frequently outside the main object.",
        "limitation": "Center filtering is only a rough proxy for object regions and does not guarantee object-focused patches.",
        "next_step": "Analyze CLS-token activations for image-level semantics.",
    },
    {
        "experiment_id": "layer12_cls_sae",
        "representation_type": "CLS token",
        "layer": "12",
        "activation_shape": "[200, 768]",
        "sae_input_unit": "one 768-d image-level CLS activation",
        "interpretation_unit": "top activating full images",
        "selection_method": "top-10 class purity",
        "main_observation": "Some CLS-token SAE features show category-related behavior.",
        "example_feature": "feature 439",
        "evidence": "Feature 439 has top10_purity = 0.8 for electric ray images.",
        "limitation": "Only one high-purity feature was found so far, and the CLS dataset has only 200 examples.",
        "next_step": "Scale to more images and inspect more CLS-token features.",
    },
], columns=[
    "experiment_id",
    "representation_type",
    "layer",
    "activation_shape",
    "sae_input_unit",
    "interpretation_unit",
    "selection_method",
    "main_observation",
    "example_feature",
    "evidence",
    "limitation",
    "next_step",
])

summary_path = project_root / "outputs" / "feature_rankings" / "current_sae_experiment_summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
experiment_summary.to_csv(summary_path, index=False)

print(f"Saved experiment summary path: {summary_path}")
print(experiment_summary)